# Qualidade de Dados e Dimensão NCM
**Projeto:** analise-importacoes-brasil  
**Fonte:** Receita Federal do Brasil — Nomenclatura Comum do Mercosul  
**Portal:** https://www.gov.br/receitafederal/pt-br/assuntos/aduana-e-comercio-exterior/classificacao-fiscal-de-mercadorias/ncm

Este notebook:
1. Lê e normaliza os três arquivos baixados da RFB
2. Executa checagens de qualidade de dados
3. Gera `data_ref/dim_ncm.csv` — dimensão única com hierarquia completa e série histórica

---
## 0. Dependências

In [4]:
# pip install pandas openpyxl

import warnings
from pathlib import Path

import pandas as pd
import numpy as np
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 20)

print('Dependências carregadas.')

Dependências carregadas.


In [2]:
PASTA_ARQUIVOS  = Path('K:/Download/dados_importacoes_temp')   # <-- ajuste se necessário
PASTA_DADOS_REF = Path('../../data_ref')                       # raiz do projeto/data_ref

ARQ_VIGENTE   = PASTA_ARQUIVOS / 'Tabela_NCM_Vigente_20261119.xlsx'
ARQ_CONCAT    = PASTA_ARQUIVOS / 'Tabela_NCM_Desc_Concatenada_Vigente_20261119.xlsx'
ARQ_HISTORICO = PASTA_ARQUIVOS / 'Tabela_NCM_Desc_Concatenada_Historica_20261119.xlsx'

DATA_CORTE     = pd.Timestamp('2022-04-01')
DATA_FIM_INDEF = pd.Timestamp('9999-12-31')

print(f'Pasta dos arquivos : {PASTA_ARQUIVOS}')
print(f'Dados de referência: {PASTA_DADOS_REF}')
print(f'Corte da série     : {DATA_CORTE.date()}')

Pasta dos arquivos : K:\Download\dados_importacoes_temp
Dados de referência: ..\..\data_ref
Corte da série     : 2022-04-01


---
## 1. Leitura e normalização dos arquivos

In [3]:
def ler_tabela_ncm(caminho: Path, renomear_duplicadas: bool = False) -> pd.DataFrame:
    """
    Lê um arquivo Excel de NCM da RFB, pula as 3 linhas de cabeçalho
    e usa a 4ª linha como nomes de coluna.
    """
    df = pd.read_excel(caminho, skiprows=3, dtype=str)
    df.columns = df.iloc[0].tolist()
    df = df.iloc[1:].reset_index(drop=True).dropna(how='all')

    if renomear_duplicadas:
        cols = df.columns.tolist()
        cols[8]  = 'Ato Legal Fim'
        cols[9]  = 'Numero Fim'
        cols[10] = 'Ano Fim'
        df.columns = cols

    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('ó', 'o', regex=False).str.replace('í', 'i', regex=False)
        .str.replace('ç', 'c', regex=False).str.replace('ã', 'a', regex=False)
    )

    for col in ['data_inicio', 'data_fim']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')

    df['codigo_limpo'] = df['codigo'].str.replace('.', '', regex=False)
    df['n_digitos']    = df['codigo_limpo'].str.len()
    return df


df_vigente   = ler_tabela_ncm(ARQ_VIGENTE)
df_concat    = ler_tabela_ncm(ARQ_CONCAT)
df_historico = ler_tabela_ncm(ARQ_HISTORICO, renomear_duplicadas=True)

print('Arquivos lidos.')
print(f'  Vigente    : {df_vigente.shape}')
print(f'  Concatenado: {df_concat.shape}')
print(f'  Histórico  : {df_historico.shape}')

Arquivos lidos.
  Vigente    : (15157, 9)
  Concatenado: (15157, 10)
  Histórico  : (269, 13)


---
## 2. Checagens de qualidade de dados

In [4]:
erros  = []
avisos = []

print('=' * 60)
print('CHECAGENS DE QUALIDADE — ARQUIVO HISTÓRICO')
print('=' * 60)

# 1. Nulos em colunas obrigatórias
for col in ['codigo', 'descricao', 'data_inicio', 'data_fim']:
    nulos = df_historico[col].isna().sum()
    if nulos > 0:
        erros.append(f'Nulos em "{col}": {nulos} linha(s)')
    else:
        print(f'✅ "{col}": sem nulos')

# 2. Código vazio
cod_vazio = (df_historico['codigo'].str.strip() == '').sum()
if cod_vazio > 0:
    erros.append(f'Códigos vazios: {cod_vazio}')
else:
    print('✅ Nenhum código vazio')

# 3. Data início posterior à data fim
inv = (df_historico['data_inicio'] > df_historico['data_fim']).sum()
if inv > 0:
    erros.append(f'data_inicio > data_fim: {inv} linha(s)')
else:
    print('✅ data_inicio <= data_fim: todas as linhas válidas')

# 4. Registros anteriores ao corte
antes_corte = (df_historico['data_inicio'] < DATA_CORTE).sum()
if antes_corte > 0:
    avisos.append(f'{antes_corte} registro(s) com data_inicio anterior a {DATA_CORTE.date()}')
else:
    print(f'✅ Todos os registros têm data_inicio >= {DATA_CORTE.date()}')

# 5. Duplicatas de código + período
dup = df_historico.duplicated(subset=['codigo', 'data_inicio', 'data_fim']).sum()
if dup > 0:
    erros.append(f'Chave (codigo + data_inicio + data_fim) duplicada: {dup} linha(s)')
else:
    print('✅ Chave (codigo + data_inicio + data_fim): sem duplicatas')

# 6. Tamanho inesperado de código
inesperados = df_historico[~df_historico['n_digitos'].isin({2, 4, 6, 8})]
if len(inesperados) > 0:
    avisos.append(f'{len(inesperados)} código(s) com tamanho fora do padrão NCM (2/4/6/8 dígitos)')
else:
    print('✅ Todos os códigos têm 2, 4, 6 ou 8 dígitos (padrão NCM)')

# 7. Inconsistência: data_fim=9999 com ato_legal_fim preenchido
incons = df_historico[
    (df_historico['n_digitos'] == 8) &
    (df_historico['data_fim'] == DATA_FIM_INDEF) &
    (df_historico['ato_legal_fim'].notna())
]
if len(incons) > 0:
    avisos.append(f'{len(incons)} NCM(s) com data_fim=9999 mas ato_legal_fim preenchido')
else:
    print('✅ Sem inconsistências entre data_fim e ato_legal_fim')

# 8. Vigente: NCMs folha sem descrição
sem_desc = (df_vigente[df_vigente['n_digitos'] == 8]['descricao'].str.strip() == '').sum()
if sem_desc > 0:
    avisos.append(f'Vigente: {sem_desc} NCM(s) folha sem descrição')
else:
    print('✅ Vigente: todos os NCMs folha têm descrição')

print('\n' + '=' * 60)
if erros:
    print(f'\n❌ ERROS ({len(erros)}):')
    for e in erros: print(f'   • {e}')
else:
    print('\n✅ Nenhum erro encontrado.')
if avisos:
    print(f'\n⚠️  AVISOS ({len(avisos)}):')
    for a in avisos: print(f'   • {a}')
else:
    print('✅ Nenhum aviso.')

CHECAGENS DE QUALIDADE — ARQUIVO HISTÓRICO
✅ "codigo": sem nulos
✅ "descricao": sem nulos
✅ "data_inicio": sem nulos
✅ "data_fim": sem nulos
✅ Nenhum código vazio
✅ data_inicio <= data_fim: todas as linhas válidas
✅ Todos os registros têm data_inicio >= 2022-04-01
✅ Chave (codigo + data_inicio + data_fim): sem duplicatas
✅ Sem inconsistências entre data_fim e ato_legal_fim
✅ Vigente: todos os NCMs folha têm descrição


✅ Nenhum erro encontrado.

⚠️  AVISOS (1):
   • 24 código(s) com tamanho fora do padrão NCM (2/4/6/8 dígitos)


---
## 3. Mapeamento das 21 Seções

As Seções não têm código numérico próprio nos arquivos da RFB. O mapeamento abaixo
é fixo e derivado da documentação oficial da NCM/SH, conforme
https://www.gov.br/receitafederal/pt-br/assuntos/aduana-e-comercio-exterior/classificacao-fiscal-de-mercadorias/ncm

In [5]:
SECOES = {
    'I'    : ('Animais vivos e produtos do reino animal',                                                                    range(1,  6)),
    'II'   : ('Produtos do reino vegetal',                                                                                   range(6,  15)),
    'III'  : ('Gorduras e óleos animais ou vegetais; produtos da sua dissociação; gorduras alimentares elaboradas',          range(15, 16)),
    'IV'   : ('Produtos das indústrias alimentares; bebidas, líquidos alcoólicos e vinagres; tabaco e seus sucedâneos manufaturados', range(16, 25)),
    'V'    : ('Produtos minerais',                                                                                           range(25, 28)),
    'VI'   : ('Produtos das indústrias químicas ou das indústrias conexas',                                                  range(28, 39)),
    'VII'  : ('Plásticos e suas obras; borracha e suas obras',                                                               range(39, 41)),
    'VIII' : ('Peles, couros, peles com pelo e obras destas matérias; artigos de seleiro e de correeiro; artigos de viagem', range(41, 44)),
    'IX'   : ('Madeira, carvão vegetal e obras de madeira; cortiça e suas obras; obras de espartaria ou de cestaria',        range(44, 47)),
    'X'    : ('Pastas de madeira ou de outras matérias fibrosas celulósicas; papel e cartão e suas obras',                   range(47, 50)),
    'XI'   : ('Matérias têxteis e suas obras',                                                                               range(50, 64)),
    'XII'  : ('Calçados, chapéus e artefatos de uso semelhante, guarda-chuvas, bengalas, vassouras, chicotes',               range(64, 68)),
    'XIII' : ('Obras de pedra, gesso, cimento, amianto, mica ou de matérias semelhantes; produtos cerâmicos; vidro e suas obras', range(68, 71)),
    'XIV'  : ('Pérolas naturais ou cultivadas, pedras preciosas ou semipreciosas, metais preciosos',                         range(71, 72)),
    'XV'   : ('Metais comuns e suas obras',                                                                                  range(72, 84)),
    'XVI'  : ('Máquinas e aparelhos, material elétrico e suas partes; aparelhos de gravação ou de reprodução de som',        range(84, 86)),
    'XVII' : ('Material de transporte',                                                                                      range(86, 90)),
    'XVIII': ('Instrumentos e aparelhos de óptica, de fotografia, de cinematografia, de medida, de controle ou de precisão', range(90, 93)),
    'XIX'  : ('Armas e munições; suas partes e acessórios',                                                                  range(93, 94)),
    'XX'   : ('Mercadorias e produtos diversos',                                                                             range(94, 97)),
    'XXI'  : ('Objetos de arte, de coleção ou de antiguidade',                                                               range(97, 100)),
}

cap_to_secao = {}
for romano, (nome, caps) in SECOES.items():
    for cap in caps:
        cap_to_secao[str(cap).zfill(2)] = {'secao_romano': romano, 'secao_nome': nome}

secao_df = pd.DataFrame([{'capitulo': k, **v} for k, v in cap_to_secao.items()])

print(f'Seções mapeadas  : {len(SECOES)}')
print(f'Capítulos cobertos: {len(cap_to_secao)}')
display(pd.DataFrame([
    {'Seção': rom, 'Nome': nome[:70], 'Caps': f"{min(caps):02d}–{max(caps):02d}"}
    for rom, (nome, caps) in SECOES.items()
]))

Seções mapeadas  : 21
Capítulos cobertos: 99


,Seção,Nome,Caps
0,I,Animais vivos e produtos do reino animal,01–05
1,II,Produtos do reino vegetal,06–14
2,III,Gorduras e óleos animais ou vegetais; produtos da sua dissociação; gor,15–15
3,IV,"Produtos das indústrias alimentares; bebidas, líquidos alcoólicos e vi",16–24
4,V,Produtos minerais,25–27
5,VI,Produtos das indústrias químicas ou das indústrias conexas,28–38
6,VII,Plásticos e suas obras; borracha e suas obras,39–40
7,VIII,"Peles, couros, peles com pelo e obras destas matérias; artigos de sele",41–43
8,IX,"Madeira, carvão vegetal e obras de madeira; cortiça e suas obras; obra",44–46
9,X,Pastas de madeira ou de outras matérias fibrosas celulósicas; papel e,47–49


---
## 4. Lookups de hierarquia

O arquivo vigente contém todos os níveis da NCM. A subposição de 6 dígitos nem sempre
existe quando um NCM tem apenas uma subposição — nesses casos, usamos o nível
intermediário de 5 dígitos como fallback, e em último caso a própria descrição do NCM.

In [6]:
# Lookup de descrição por nível (do arquivo vigente)
desc_cap  = df_vigente[df_vigente['n_digitos']==2].set_index('codigo_limpo')['descricao']  # capítulo
desc_pos  = df_vigente[df_vigente['n_digitos']==4].set_index('codigo_limpo')['descricao']  # posição
desc_sub5 = df_vigente[df_vigente['n_digitos']==5].set_index('codigo_limpo')['descricao']  # sub intermediária
desc_sub6 = df_vigente[df_vigente['n_digitos']==6].set_index('codigo_limpo')['descricao']  # subposição SH
desc_conc = df_concat[df_concat['n_digitos']==8].set_index('codigo_limpo')['descricao_concatenada']  # concatenada

print(f'Capítulos   : {len(desc_cap)}')
print(f'Posições    : {len(desc_pos)}')
print(f'Sub (5d)    : {len(desc_sub5)}')
print(f'Sub (6d/SH) : {len(desc_sub6)}')
print(f'Concatenadas: {len(desc_conc)}')

Capítulos   : 96
Posições    : 957
Sub (5d)    : 1024
Sub (6d/SH) : 1661
Concatenadas: 10516


---
## 5. Composição da dimensão única

A `dim_ncm.csv` reúne três grupos em uma única tabela:

| Status | Origem | Descrição |
|---|---|---|
| `encerrado` | Histórico | NCMs que deixaram de vigorar desde abril/2022 |
| `vigente_alterado` | Histórico | NCMs que passaram a vigorar no período (substituíram encerrados) |
| `vigente_estavel` | Vigente | NCMs sem nenhuma alteração desde abril/2022 |

In [7]:
# GRUPOS 1 e 2 — histórico, apenas NCMs folha (8 dígitos)
hist_folha = df_historico[df_historico['n_digitos'] == 8].copy()
hist_folha['encerrado'] = hist_folha['data_fim'] < DATA_FIM_INDEF
hist_folha['status']    = hist_folha['encerrado'].map({True: 'encerrado', False: 'vigente_alterado'})
hist_folha['origem']    = 'historico'

# GRUPO 3 — estáveis (vigente, 8 dígitos, fora do histórico)
ncms_no_hist = set(hist_folha['codigo_limpo'])
vig_folha    = df_vigente[df_vigente['n_digitos'] == 8].copy()
estaveis     = vig_folha[~vig_folha['codigo_limpo'].isin(ncms_no_hist)].copy()
estaveis['encerrado']             = False
estaveis['status']                = 'vigente_estavel'
estaveis['origem']                = 'vigente'
estaveis['descricao_concatenada'] = estaveis['codigo_limpo'].map(desc_conc)
estaveis['ato_legal_fim']         = None
estaveis['numero_fim']            = None
estaveis['ano_fim']               = None

print(f'Grupo 1 — encerrados        : {hist_folha["encerrado"].sum()}')
print(f'Grupo 2 — vigentes alterados: {(~hist_folha["encerrado"]).sum()}')
print(f'Grupo 3 — vigentes estáveis : {len(estaveis)}')

Grupo 1 — encerrados        : 71
Grupo 2 — vigentes alterados: 153
Grupo 3 — vigentes estáveis : 10363


In [8]:
def preparar_grupo(df: pd.DataFrame, origem: str) -> pd.DataFrame:
    """Padroniza as colunas de cada grupo para o schema final da dim_ncm."""
    out = pd.DataFrame(index=df.index)
    out['ncm']                   = df['codigo_limpo']
    out['ncm_formatado']         = df['codigo']
    out['descricao']             = df['descricao']
    out['descricao_concatenada'] = df.get('descricao_concatenada', pd.Series(dtype=str, index=df.index))
    out['capitulo']              = df['codigo_limpo'].str[:2]
    out['posicao']               = df['codigo_limpo'].str[:4]
    out['subposicao']            = df['codigo_limpo'].str[:6]
    out['vigencia_inicio']       = df['data_inicio']
    out['vigencia_fim']          = df['data_fim']
    out['status']                = df['status']
    out['encerrado']             = df['encerrado']
    out['origem']                = origem
    out['ato_legal_inicio']      = df.get('ato_legal_inicio', pd.Series(dtype=str, index=df.index))
    out['numero_inicio']         = df.get('número',    pd.Series(dtype=str, index=df.index))
    out['ano_inicio']            = df.get('ano',       pd.Series(dtype=str, index=df.index))
    out['ato_legal_fim']         = df.get('ato_legal_fim', pd.Series(dtype=str, index=df.index))
    out['numero_fim']            = df.get('numero_fim', pd.Series(dtype=str, index=df.index))
    out['ano_fim']               = df.get('ano_fim',   pd.Series(dtype=str, index=df.index))
    return out


dim_ncm = pd.concat([
    preparar_grupo(hist_folha, 'historico'),
    preparar_grupo(estaveis,   'vigente'),
], ignore_index=True)

print(f'dim_ncm antes da hierarquia: {dim_ncm.shape}')

dim_ncm antes da hierarquia: (10587, 18)


---
## 6. Enriquecimento com hierarquia

Estratégia de preenchimento da `subposicao_descricao`:
1. Nível 6 dígitos (subposição SH padrão) — cobre 62% dos NCMs
2. Nível 5 dígitos (subposição intermediária) — cobre mais 21%
3. Nível 4 dígitos (posição) — para NCMs de posição única sem desdobramento
4. Descrição do próprio NCM — último fallback para ~1,6% residual

In [9]:
sub5_col = dim_ncm['subposicao'].str[:5]  # prefixo de 5 dígitos

# subposicao_descricao: fallback em cascata
dim_ncm['subposicao_descricao'] = (
    dim_ncm['subposicao'].map(desc_sub6)         # nível 6 (SH)
    .fillna(sub5_col.map(desc_sub5))             # nível 5 (intermediário)
    .fillna(dim_ncm['posicao'].map(desc_pos))    # posição (4d)
    .fillna(dim_ncm['descricao'])                # próprio NCM (último fallback)
)

# Colunas de hierarquia
dim_ncm['posicao_descricao']  = dim_ncm['posicao'].map(desc_pos)
dim_ncm['capitulo_descricao'] = dim_ncm['capitulo'].map(desc_cap)

# Seção (via mapa fixo)
dim_ncm = dim_ncm.merge(secao_df, on='capitulo', how='left')

# Descrição concatenada para NCMs do histórico que não têm no arquivo concatenado
sem_conc = dim_ncm['descricao_concatenada'].isna()
dim_ncm.loc[sem_conc, 'descricao_concatenada'] = dim_ncm.loc[sem_conc, 'ncm'].map(desc_conc)

# Ordenar colunas
COLS_FINAIS = [
    'ncm', 'ncm_formatado', 'descricao', 'descricao_concatenada',
    'subposicao', 'subposicao_descricao',
    'posicao',    'posicao_descricao',
    'capitulo',   'capitulo_descricao',
    'secao_romano', 'secao_nome',
    'vigencia_inicio', 'vigencia_fim',
    'status', 'encerrado', 'origem',
    'ato_legal_inicio', 'numero_inicio', 'ano_inicio',
    'ato_legal_fim',    'numero_fim',    'ano_fim',
]
dim_ncm = dim_ncm[COLS_FINAIS].sort_values(['ncm', 'vigencia_inicio']).reset_index(drop=True)

print(f'dim_ncm final: {dim_ncm.shape}')
display(dim_ncm.head(5))

dim_ncm final: (10587, 23)


,ncm,ncm_formatado,descricao,descricao_concatenada,subposicao,subposicao_descricao,posicao,posicao_descricao,capitulo,capitulo_descricao,...,vigencia_fim,status,encerrado,origem,ato_legal_inicio,numero_inicio,ano_inicio,ato_legal_fim,numero_fim,ano_fim
0,01012100,0101.21.00,-- Reprodutores de raça pura,"01 Animais vivos. \n01.01 Cavalos, asininos e muares, vivos. \n0101.2 - Cava...",010121,- Cavalos:,0101,"Cavalos, asininos e muares, vivos.",01,Animais vivos.,...,9999-12-31,vigente_estavel,False,vigente,Res Gecex,272,2021,None,None,None
1,01012900,0101.29.00,-- Outros,"01 Animais vivos. \n01.01 Cavalos, asininos e muares, vivos. \n0101.2 - Cava...",010129,- Cavalos:,0101,"Cavalos, asininos e muares, vivos.",01,Animais vivos.,...,9999-12-31,vigente_estavel,False,vigente,Res Gecex,272,2021,None,None,None
2,01013000,0101.30.00,- Asininos,"01 Animais vivos. \n01.01 Cavalos, asininos e muares, vivos. \n0101.30.00 - ...",010130,"Cavalos, asininos e muares, vivos.",0101,"Cavalos, asininos e muares, vivos.",01,Animais vivos.,...,9999-12-31,vigente_estavel,False,vigente,Res Gecex,272,2021,None,None,None
3,01019000,0101.90.00,- Outros,"01 Animais vivos. \n01.01 Cavalos, asininos e muares, vivos. \n0101.90.00 - ...",010190,"Cavalos, asininos e muares, vivos.",0101,"Cavalos, asininos e muares, vivos.",01,Animais vivos.,...,9999-12-31,vigente_estavel,False,vigente,Res Gecex,272,2021,None,None,None
4,01022110,0102.21.10,Prenhes ou com cria ao pé,01 Animais vivos. \n01.02 Animais vivos da espécie bovina. \n0102.2 - Bovino...,010221,-- Reprodutores de raça pura,0102,Animais vivos da espécie bovina.,01,Animais vivos.,...,9999-12-31,vigente_estavel,False,vigente,Res Gecex,272,2021,None,None,None


---
## 7. Checagens de qualidade da dim_ncm

In [10]:
erros_dim  = []
avisos_dim = []

print('=' * 60)
print('CHECAGENS DE QUALIDADE — dim_ncm')
print('=' * 60)

# 1. NCM com tamanho != 8
inv = (dim_ncm['ncm'].str.len() != 8).sum()
if inv > 0: erros_dim.append(f'NCMs com tamanho != 8: {inv}')
else: print('✅ Todos os NCMs com 8 dígitos')

# 2. Nulos em colunas essenciais
for col in ['ncm', 'descricao', 'capitulo', 'secao_romano', 'vigencia_inicio', 'vigencia_fim']:
    nulos = dim_ncm[col].isna().sum()
    if nulos > 0: erros_dim.append(f'Nulos em "{col}": {nulos}')
    else: print(f'✅ "{col}": sem nulos')

# 3. Consistência capítulo
incons = (dim_ncm['ncm'].str[:2] != dim_ncm['capitulo']).sum()
if incons > 0: erros_dim.append(f'Capítulo inconsistente: {incons} linha(s)')
else: print('✅ Capítulo consistente com os 2 primeiros dígitos do NCM')

# 4. vigencia_inicio <= vigencia_fim
inv_datas = (dim_ncm['vigencia_inicio'] > dim_ncm['vigencia_fim']).sum()
if inv_datas > 0: erros_dim.append(f'vigencia_inicio > vigencia_fim: {inv_datas}')
else: print('✅ vigencia_inicio <= vigencia_fim: todas as linhas válidas')

# 5. Avisos — nulos tolerados
for col, label in [
    ('subposicao_descricao', 'subposicao_descricao'),
    ('posicao_descricao',    'posicao_descricao'),
    ('descricao_concatenada','descricao_concatenada'),
]:
    nulos = dim_ncm[col].isna().sum()
    if nulos > 0: avisos_dim.append(f'Nulos em "{label}": {nulos} linha(s)')
    else: print(f'✅ "{label}": sem nulos')

# 6. Resumo por status
print(f'\n📊 NCMs por status:')
display(dim_ncm.groupby('status').agg(
    registros=('ncm','count'),
    ncms_unicos=('ncm','nunique')
).reset_index())

# 7. NCMs com múltiplos registros (encerrado + substituto)
mult = dim_ncm.groupby('ncm').size()
mult_registros = mult[mult > 1]
print(f'\nNCMs com múltiplos registros (mudança de vigência): {len(mult_registros)}')
if len(mult_registros) > 0:
    ex = mult_registros.index[0]
    print(f'\nExemplo — NCM {ex}:')
    display(dim_ncm[dim_ncm['ncm'] == ex][
        ['ncm', 'descricao', 'vigencia_inicio', 'vigencia_fim', 'status']
    ])

# Relatório
print('\n' + '=' * 60)
if erros_dim:
    print(f'\n❌ ERROS ({len(erros_dim)}):')
    for e in erros_dim: print(f'   • {e}')
else:
    print('\n✅ Nenhum erro encontrado.')
if avisos_dim:
    print(f'\n⚠️  AVISOS ({len(avisos_dim)}):')
    for a in avisos_dim: print(f'   • {a}')
else:
    print('✅ Nenhum aviso.')

CHECAGENS DE QUALIDADE — dim_ncm
✅ Todos os NCMs com 8 dígitos
✅ "ncm": sem nulos
✅ "descricao": sem nulos
✅ "capitulo": sem nulos
✅ "secao_romano": sem nulos
✅ "vigencia_inicio": sem nulos
✅ "vigencia_fim": sem nulos
✅ Capítulo consistente com os 2 primeiros dígitos do NCM
✅ vigencia_inicio <= vigencia_fim: todas as linhas válidas
✅ "subposicao_descricao": sem nulos
✅ "descricao_concatenada": sem nulos

📊 NCMs por status:


,status,registros,ncms_unicos
0,encerrado,71,70
1,vigente_alterado,153,153
2,vigente_estavel,10363,10363



NCMs com múltiplos registros (mudança de vigência): 18

Exemplo — NCM 21069030:


,ncm,descricao,vigencia_inicio,vigencia_fim,status
1272,21069030,Complementos alimentares,2022-04-01,2023-10-31,encerrado
1273,21069030,Suplementos alimentares,2023-11-01,9999-12-31,vigente_alterado




✅ Nenhum erro encontrado.

⚠️  AVISOS (1):
   • Nulos em "posicao_descricao": 483 linha(s)


In [12]:
# Quantidade de valores ausentes
nulos = dim_ncm['posicao_descricao'].isna().sum()

# Percentual de valores ausentes
percentual = dim_ncm['posicao_descricao'].isna().mean() * 100

print(f"Valores nulos: {nulos}")
print(f"Percentual de nulos: {percentual:.2f}%")

# Identificar os registros afetados
dim_ncm.loc[
    dim_ncm['posicao_descricao'].isna(),
    ['ncm', 'posicao', 'posicao_descricao']
]

Valores nulos: 483
Percentual de nulos: 4.56%


,ncm,posicao,posicao_descricao
70,02050000,0205,NaN
513,04090000,0409,NaN
516,05010000,0501,NaN
522,05040011,0504,NaN
523,05040012,0504,NaN
...,...,...,...
10564,96170020,9617,NaN
10565,96180000,9618,NaN
10566,96190000,9619,NaN
10567,96200000,9620,NaN


---
## 8. Exploração da dimensão

Se torna notável o quão granular pode chegar um número de NCM e é necessário ter um certo cuidado ao tomar conclusão para níveis maiores de granularidade.

In [11]:
# Cobertura por seção
print('NCMs por seção:')
display(
    dim_ncm[dim_ncm['status'] != 'encerrado']
    .groupby(['secao_romano', 'secao_nome'])
    .size().reset_index(name='n_ncms')
    .sort_values('secao_romano')
)

NCMs por seção:


,secao_romano,secao_nome,n_ncms
0,I,Animais vivos e produtos do reino animal,539
1,II,Produtos do reino vegetal,420
2,III,Gorduras e óleos animais ou vegetais; produtos da sua dissociação; gorduras ...,81
3,IV,"Produtos das indústrias alimentares; bebidas, líquidos alcoólicos e vinagres...",323
4,IX,"Madeira, carvão vegetal e obras de madeira; cortiça e suas obras; obras de e...",182
5,V,Produtos minerais,211
6,VI,Produtos das indústrias químicas ou das indústrias conexas,3116
7,VII,Plásticos e suas obras; borracha e suas obras,441
8,VIII,"Peles, couros, peles com pelo e obras destas matérias; artigos de seleiro e ...",113
9,X,Pastas de madeira ou de outras matérias fibrosas celulósicas; papel e cartão...,226


In [12]:
# NCMs encerrados — amostra com hierarquia completa
print('Amostra de NCMs encerrados:')
display(
    dim_ncm[dim_ncm['status'] == 'encerrado'][
        ['ncm', 'descricao', 'capitulo', 'secao_romano',
         'vigencia_inicio', 'vigencia_fim', 'ato_legal_fim']
    ].head(10)
)

Amostra de NCMs encerrados:


,ncm,descricao,capitulo,secao_romano,vigencia_inicio,vigencia_fim,ato_legal_fim
82,02071200,"-- Não cortadas em pedaços, congeladas",02,I,2022-04-01,2023-03-31,Res Gecex
86,02071400,"-- Pedaços e miudezas, congelados",02,I,2022-04-01,2024-07-31,Res Gecex
223,03029100,"-- Fígados, ovas e gônadas masculinas",03,I,2022-04-01,2023-03-31,Res Gecex
302,03039100,"-- Fígados, ovas e gônadas masculinas",03,I,2022-04-01,2023-03-31,Res Gecex
371,03052000,"- Fígados, ovas e gônadas masculinas, de peixes, secos, defumados (fumados),...",03,I,2022-04-01,2023-03-31,Res Gecex
630,07129010,Alho em pó,07,II,2022-04-01,2024-12-31,Res Gecex
1007,15132110,De amêndoa de palma (palmiste) (coconote),15,III,2022-04-01,2022-06-30,Res Gecex
1011,15132910,De amêndoa de palma (palmiste) (coconote),15,III,2022-04-01,2022-06-30,Res Gecex
1139,19012000,"- Misturas e pastas para a preparação de produtos de padaria, pastelaria e d...",19,IV,2022-04-01,2023-12-31,Res Gecex
1272,21069030,Complementos alimentares,21,IV,2022-04-01,2023-10-31,Res Gecex


In [13]:
# NCMs com histórico de alteração — encerrado + substituto lado a lado
ncms_alterados = dim_ncm[dim_ncm['ncm'].duplicated(keep=False)]['ncm'].unique()
print(f'NCMs com mais de um registro (alterações): {len(ncms_alterados)}')
print('\nExemplo de NCM que foi encerrado e substituído:')
ex_ncm = ncms_alterados[0]
display(
    dim_ncm[dim_ncm['ncm'] == ex_ncm][
        ['ncm','descricao','vigencia_inicio','vigencia_fim',
         'status','ato_legal_inicio','ato_legal_fim']
    ]
)

NCMs com mais de um registro (alterações): 18

Exemplo de NCM que foi encerrado e substituído:


,ncm,descricao,vigencia_inicio,vigencia_fim,status,ato_legal_inicio,ato_legal_fim
1272,21069030,Complementos alimentares,2022-04-01,2023-10-31,encerrado,Res Gecex,Res Gecex
1273,21069030,Suplementos alimentares,2023-11-01,9999-12-31,vigente_alterado,Res Gecex,NaN


### NCM de Consumos Recentes Meus
Compras recentes de itens no Aliexpress. Será se estão presentes na dimensão?

| Produto | Capítulo | Posição | Exemplo de NCM |
|---|---|---|---|
| Power bank de íon de lítio | 85 | 8507 — Acumuladores elétricos | `85076000` |
| Cabo USB com conectores | 85 | 8544 — Fios e cabos elétricos isolados | `85444200` |
| Celular smartphone | 85 | 8517 — Aparelhos de comunicação | `85171300` |
| Console de videogame | 95 | 9504 — Jogos e artigos para jogos | `95045000` |

In [21]:
print('Acumuladores elétricos')
dim_ncm[dim_ncm['posicao'] == '8507'][['ncm','descricao','vigencia_inicio','vigencia_fim','status']] 

Acumuladores elétricos


,ncm,descricao,vigencia_inicio,vigencia_fim,status
9115,85071010,De capacidade inferior ou igual a 20 Ah e tensão inferior ou igual a 12 V,2022-04-01,9999-12-31,vigente_estavel
9116,85071090,Outros,2022-04-01,9999-12-31,vigente_estavel
9117,85072010,De peso inferior ou igual a 1.000 kg,2022-04-01,9999-12-31,vigente_estavel
9118,85072090,Outros,2022-04-01,9999-12-31,vigente_estavel
9119,85073011,De capacidade inferior ou igual a 15 Ah,2022-04-01,9999-12-31,vigente_estavel
9120,85073019,Outros,2022-04-01,9999-12-31,vigente_estavel
9121,85073090,Outros,2022-04-01,9999-12-31,vigente_estavel
9122,85075010,"De tensão igual a 1,2 V, cilíndricos do tipo HR6 (AA)",2022-04-01,9999-12-31,vigente_estavel
9123,85075020,"De tensão igual a 1,2 V, cilíndricos do tipo HR03 (AAA)",2022-04-01,9999-12-31,vigente_estavel
9124,85075090,Outros,2022-04-01,9999-12-31,vigente_estavel


In [22]:
print('Fios e cabos isolados')
dim_ncm[dim_ncm['posicao'] == '8544'][['ncm','descricao','vigencia_inicio','vigencia_fim','status']] 

Fios e cabos isolados


,ncm,descricao,vigencia_inicio,vigencia_fim,status
9607,85441100,-- De cobre,2022-04-01,9999-12-31,vigente_estavel
9608,85441910,De alumínio,2022-04-01,2024-03-31,encerrado
9609,85441911,Revestido de cobre (CCA - Copper Clad Aluminum),2024-04-01,9999-12-31,vigente_alterado
9610,85441919,Outros,2024-04-01,9999-12-31,vigente_alterado
9611,85441990,Outros,2022-04-01,9999-12-31,vigente_estavel
9612,85442000,- Cabos coaxiais e outros condutores elétricos coaxiais,2022-04-01,9999-12-31,vigente_estavel
9613,85443000,- Jogos de fios para velas de ignição e outros jogos de fios do tipo utiliza...,2022-04-01,9999-12-31,vigente_estavel
9614,85444200,-- Munidos de peças de conexão,2022-04-01,9999-12-31,vigente_estavel
9615,85444900,-- Outros,2022-04-01,9999-12-31,vigente_estavel
9616,85446000,"- Outros condutores elétricos, para uma tensão superior a 1.000 V",2022-04-01,9999-12-31,vigente_estavel


In [23]:
print('Aparelhos de comunicação')
dim_ncm[dim_ncm['posicao'] == '8517'][['ncm','descricao','vigencia_inicio','vigencia_fim','status']]

Aparelhos de comunicação


,ncm,descricao,vigencia_inicio,vigencia_fim,status
9214,85171100,-- Aparelhos telefônicos por fio com unidade auscultador-microfone sem fio,2022-04-01,9999-12-31,vigente_estavel
9215,85171300,-- Telefones inteligentes (smartphones),2022-04-01,9999-12-31,vigente_estavel
9216,85171410,"De radiotelefonia, analógicos",2022-04-01,9999-12-31,vigente_estavel
9217,85171431,Portáteis,2022-04-01,9999-12-31,vigente_estavel
9218,85171432,"Fixos, sem fonte própria de energia",2022-04-01,9999-12-31,vigente_estavel
9219,85171439,Outros,2022-04-01,9999-12-31,vigente_estavel
9220,85171441,"Digitais, operando em banda C, Ku, L ou S",2022-04-01,9999-12-31,vigente_estavel
9221,85171449,Outros,2022-04-01,9999-12-31,vigente_estavel
9222,85171490,Outros,2022-04-01,9999-12-31,vigente_estavel
9223,85171830,Não combinados com outros aparelhos,2022-04-01,9999-12-31,vigente_estavel


In [24]:
print('Jogos e artigos para jogos')
dim_ncm[dim_ncm['posicao'] == '9504'][['ncm','descricao','vigencia_inicio','vigencia_fim','status']] 

Jogos e artigos para jogos


,ncm,descricao,vigencia_inicio,vigencia_fim,status
10471,95042000,- Bilhares de qualquer espécie e seus acessórios,2022-04-01,9999-12-31,vigente_estavel
10472,95043000,"- Outros jogos que funcionem por introdução de moedas, notas (papéis-moeda),...",2022-04-01,9999-12-31,vigente_estavel
10473,95044000,- Cartas de jogar,2022-04-01,9999-12-31,vigente_estavel
10474,95045000,"- Consoles e máquinas de jogos de vídeo, exceto os classificados na subposiç...",2022-04-01,9999-12-31,vigente_estavel
10475,95049010,Jogos de balizas (pinos) automáticos (boliche),2022-04-01,9999-12-31,vigente_estavel
10476,95049090,Outros,2022-04-01,9999-12-31,vigente_estavel


---
## 9. Salvar `dim_ncm.csv`

In [14]:
PASTA_DADOS_REF.mkdir(parents=True, exist_ok=True)
caminho_saida = PASTA_DADOS_REF / 'dim_ncm.csv'

dim_ncm.to_csv(
    caminho_saida,
    index=False,
    encoding='utf-8',
    date_format='%Y-%m-%d',
)

print(f'✅ Arquivo salvo em: {caminho_saida}')
print(f'   Linhas  : {len(dim_ncm):,}')
print(f'   Colunas : {dim_ncm.columns.tolist()}')
print(f'\nResumo final:')
print(f'   NCMs únicos         : {dim_ncm["ncm"].nunique():,}')
print(f'   vigente_estavel     : {(dim_ncm["status"]=="vigente_estavel").sum():,}')
print(f'   vigente_alterado    : {(dim_ncm["status"]=="vigente_alterado").sum():,}')
print(f'   encerrado           : {(dim_ncm["status"]=="encerrado").sum():,}')
print(f'   Período coberto     : {dim_ncm["vigencia_inicio"].min().date()} → {dim_ncm["vigencia_fim"].max().date()}')

✅ Arquivo salvo em: ..\..\data_ref\dim_ncm.csv
   Linhas  : 10,587
   Colunas : ['ncm', 'ncm_formatado', 'descricao', 'descricao_concatenada', 'subposicao', 'subposicao_descricao', 'posicao', 'posicao_descricao', 'capitulo', 'capitulo_descricao', 'secao_romano', 'secao_nome', 'vigencia_inicio', 'vigencia_fim', 'status', 'encerrado', 'origem', 'ato_legal_inicio', 'numero_inicio', 'ano_inicio', 'ato_legal_fim', 'numero_fim', 'ano_fim']

Resumo final:
   NCMs únicos         : 10,568
   vigente_estavel     : 10,363
   vigente_alterado    : 153
   encerrado           : 71
   Período coberto     : 2022-04-01 → 9999-12-31


---
## 10. Como usar a `dim_ncm.csv` no pipeline

### Join com a silver — hierarquia por período de vigência

```python
import pandas as pd

dim = pd.read_csv(
    'data_ref/dim_ncm.csv',
    dtype={'ncm': str, 'capitulo': str, 'posicao': str, 'subposicao': str},
    parse_dates=['vigencia_inicio', 'vigencia_fim'],
)

silver = pd.read_parquet('s3://importacoes-brasil-bucket/silver/estatisticas_di/...')
silver['ano_mes_dt'] = pd.to_datetime(silver['ano_mes'], format='%Y-%m')

# Join com tolerância de vigência — garante a descrição correta por período
silver = silver.merge(dim, on='ncm', how='left')
silver = silver.query('vigencia_inicio <= ano_mes_dt <= vigencia_fim')
```

### Drill-down no Power BI

```
dim_ncm
  └── secao_romano / secao_nome          ← nível mais alto
        └── capitulo / capitulo_descricao
              └── posicao / posicao_descricao
                    └── subposicao / subposicao_descricao
                          └── ncm / ncm_descricao    ← nível folha
```

Para análise temporal, filtre `status != 'encerrado'` para ficar apenas com NCMs
ativos em cada período. Para detectar NCMs que aparecem na silver após encerramento:

```python
enc = dim[dim['status'] == 'encerrado'][['ncm', 'vigencia_fim']]
silver_check = silver.merge(enc, on='ncm', how='inner')
problemas = silver_check[silver_check['ano_mes_dt'] > silver_check['vigencia_fim']]
print(f'Registros na silver com NCM após encerramento: {len(problemas)}')
```

> **Nota:** a coluna `subposicao_descricao` usa fallback em cascata (nível 6 → 5 → descrição do NCM)
> porque nem todo NCM tem uma subposição distinta no arquivo da RFB. Isso é estrutura normal do SH
> e não é um erro de qualidade.